# 📘 Módulo 03: JOINs Complejos, Operadores de Conjuntos & Subconsultas Combinadas
### 🔗 Dominio Completo de INNER, LEFT, RIGHT, FULL OUTER, CROSS y SELF JOIN + Combinaciones con Subconsultas y LIKE
**Asignatura:** Bases de Datos Avanzadas  
**Entorno:** Google Colab / DuckDB (100% In-Memory)  
---
## 🎯 Objetivos de Aprendizaje
1. Comprender la fundamentación algebraica y comportamiento de **todos los tipos de JOIN**:
   - `INNER JOIN` (Intersección $A \cap B$)
   - `LEFT JOIN` (Preservación de la Tabla Izquierda)
   - `RIGHT JOIN` (Preservación de la Tabla Derecha)
   - `FULL OUTER JOIN` (Unión de Conjuntos $A \cup B$)
   - `CROSS JOIN` (Producto Cartesiano $A \times B$)
   - `SELF JOIN` (Autovinculación Jerárquica $R \bowtie R$)
   - `ANTI JOIN` (Diferencia de Conjuntos $A - B$ mediante `LEFT JOIN ... WHERE B.key IS NULL`)
2. Implementar **JOINs con Subconsultas Derivadas** en la cláusula `FROM` / `JOIN`.
3. Construir **JOINs con Subconsultas filtradas por patrones `LIKE` / `ILIKE`** para resolver preguntas avanzadas de negocio.
4. Practicar operadores de conjuntos compatibles en unión (`UNION`, `UNION ALL`, `INTERSECT`, `EXCEPT`).

In [ ]:
# ==============================================================================
# 🚀 SETUP INICIAL AUTO-CONTENIDO: DUCKDB & GOOGLE COLAB (100% IN-MEMORY)
# ==============================================================================
!pip install duckdb duckdb-engine jupysql -q

import duckdb

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# Conectar a motor DuckDB en memoria
conn = duckdb.connect(database=':memory:')

# Poblar esquema de datos de referencia (Empresa & Cartera)
conn.execute("""
CREATE TABLE IF NOT EXISTS departamentos (
    depto_id INT PRIMARY KEY,
    nombre VARCHAR,
    presupuesto DECIMAL(15,2),
    ciudad VARCHAR
);
INSERT INTO departamentos VALUES
(1, 'Tecnología e Innovación', 150000000.00, 'Bogotá'),
(2, 'Ventas y Comercio', 200000000.00, 'Medellín'),
(3, 'Operaciones y Logística', 120000000.00, 'Cali'),
(4, 'Recursos Humanos', 80000000.00, 'Bogotá'),
(5, 'Investigación & I+D', 95000000.00, 'Bucaramanga'); -- Sin empleados asignados

CREATE TABLE IF NOT EXISTS empleados (
    empleado_id INT PRIMARY KEY,
    nombre VARCHAR,
    salario DECIMAL(15,2),
    jefe_id INT,
    depto_id INT
);
INSERT INTO empleados VALUES
(101, 'Carlos Mendoza (CEO)', 12000000.00, NULL, 1),
(102, 'Ana María Gómez', 6200000.00, 101, 1),
(103, 'Luis Fernando Torres', 4800000.00, 101, 1),
(104, 'Sofía Ramírez', 9000000.00, 101, 2),
(105, 'Diego Alejandro Silva', 5500000.00, 104, 2),
(106, 'Valentina Morales', 5100000.00, 104, 2),
(107, 'Javier Castro (Consultor Externo)', 7000000.00, NULL, NULL); -- Sin departamento asignado

CREATE TABLE IF NOT EXISTS clientes_cartera (
    cliente_id INT PRIMARY KEY,
    nombre VARCHAR NOT NULL,
    doc_identidad VARCHAR UNIQUE NOT NULL,
    ciudad VARCHAR,
    email VARCHAR
);
INSERT INTO clientes_cartera VALUES
(1, 'Inversiones Empresariales Bogotá S.A.S.', '900123456-1', 'Bogotá', 'contacto@inversionesbogota.co'),
(2, 'Construcciones del Valle Ltda.', '890987654-3', 'Cali', 'gerencia@convalle.com'),
(3, 'María Fernanda Gómez', '1018456789', 'Medellín', 'mfgomez@email.com'),
(4, 'Comercializadora Caribe S.A.', '800555444-2', 'Barranquilla', 'ventas@caribesa.co'),
(5, 'Distribuidora Andina S.A.S.', '901555666-4', 'Bogotá', 'contacto@andinasas.co'); -- Sin créditos

CREATE TABLE IF NOT EXISTS creditos (
    credito_id INT PRIMARY KEY,
    cliente_id INT REFERENCES clientes_cartera(cliente_id),
    monto_desembolsado DECIMAL(15,2),
    saldo_actual DECIMAL(15,2),
    estado VARCHAR
);
INSERT INTO creditos VALUES
(101, 1, 100000000.00, 75000000.00, 'VIGENTE'),
(102, 1, 40000000.00, 38000000.00, 'VIGENTE'),
(103, 2, 60000000.00, 60000000.00, 'MORA'),
(104, 3, 15000000.00, 0.00, 'PAGADO'),
(105, 4, 30000000.00, 32500000.00, 'MORA');
""")

%sql conn
print('✅ Entorno DuckDB inicializado para el Módulo 03 (JOINs Complejos).')

## 1. Catálogo Completo de Tipos de JOIN
Los JOINs permiten combinar filas de dos o más tablas basándose en una condición de relación lógica (generalmente entre Clave Primaria y Clave Foránea).
```text
flowchart LR
    subgraph INNER [INNER JOIN]
        I["Solo coincidencia en A y B (A ∩ B)"]
    end
    subgraph LEFT [LEFT JOIN]
        L["Todo A + coincidencias de B"]
    end
    subgraph RIGHT [RIGHT JOIN]
        R["Todo B + coincidencias de A"]
    end
    subgraph FULL [FULL OUTER JOIN]
        F["Todo A + Todo B (A ∪ B)"]
    end
```

### 1.1 `INNER JOIN` (Intersección $A \cap B$)
Retorna exclusivamente las tuplas que satisfacen la condición de igualdad en ambas tablas.

In [ ]:
%%sql
-- 1.1 INNER JOIN: Empleados con su departamento asignado
SELECT 
    e.empleado_id,
    e.nombre AS empleado,
    e.salario,
    d.nombre AS departamento,
    d.ciudad
FROM empleados e
INNER JOIN departamentos d ON e.depto_id = d.depto_id;

### 1.2 `LEFT JOIN` (Preservación de la Tabla Izquierda)
Retorna **todas las filas de la tabla izquierda** (`empleados`), completando con `NULL` si no existe coincidencia en la derecha.

In [ ]:
%%sql
-- 1.2 LEFT JOIN: Incluye al consultor externo Javier Castro aunque no tenga depto_id
SELECT 
    e.empleado_id,
    e.nombre AS empleado,
    COALESCE(d.nombre, 'SIN DEPARTAMENTO (EXTERNO)') AS departamento
FROM empleados e
LEFT JOIN departamentos d ON e.depto_id = d.depto_id;

### 1.3 `RIGHT JOIN` (Preservación de la Tabla Derecha)
Retorna **todas las filas de la tabla derecha** (`departamentos`), incluyendo los departamentos sin empleados asignados (ej. Investigación & I+D).

In [ ]:
%%sql
-- 1.3 RIGHT JOIN: Muestra el departamento de I+D aunque no tenga empleados asociados
SELECT 
    d.depto_id,
    d.nombre AS departamento,
    COALESCE(e.nombre, 'SIN EMPLEADOS ASIGNADOS') AS empleado
FROM empleados e
RIGHT JOIN departamentos d ON e.depto_id = d.depto_id;

### 1.4 `FULL OUTER JOIN` (Unión Completa $A \cup B$)
Combina los resultados de `LEFT JOIN` y `RIGHT JOIN`. Muestra empleados sin departamento y departamentos sin empleados.

In [ ]:
%%sql
-- 1.4 FULL OUTER JOIN: Unión completa de ambos universos con nulos en ambas direcciones
SELECT 
    COALESCE(e.nombre, 'SIN EMPLEADO') AS empleado,
    COALESCE(d.nombre, 'SIN DEPARTAMENTO') AS departamento
FROM empleados e
FULL OUTER JOIN departamentos d ON e.depto_id = d.depto_id;

### 1.5 `CROSS JOIN` (Producto Cartesiano $A \times B$)
Combina cada fila de la primera tabla con todas las filas de la segunda tabla. Si $A$ tiene 7 filas y $B$ tiene 5 filas, retorna $7 \times 5 = 35$ combinaciones.

In [ ]:
%%sql
-- 1.5 CROSS JOIN: Producto cartesiano de Empleados x Departamentos
SELECT e.nombre AS empleado, d.nombre AS departamento
FROM empleados e
CROSS JOIN departamentos d
LIMIT 10;

### 1.6 `SELF JOIN` (Autovinculación de la Misma Tabla)
Permite relacionar una tabla consigo misma mediante alias diferentes (`e` para subordinado, `jefe` para el supervisor) usando `jefe_id`.

In [ ]:
%%sql
-- 1.6 SELF JOIN: Mapeo de la jerarquía organizacional (Empleado -> Su Jefe)
SELECT 
    sub.empleado_id,
    sub.nombre AS subordinado,
    sub.salario AS salario_subordinado,
    COALESCE(jefe.nombre, 'NIVEL SUPERIOR (NO TIENE JEFE)') AS jefe_directo
FROM empleados sub
LEFT JOIN empleados jefe ON sub.jefe_id = jefe.empleado_id;

### 1.7 `ANTI JOIN` (Diferencia de Conjuntos $A - B$)
Identifica elementos en la tabla izquierda que **NO tienen correspondencia** en la tabla derecha.

In [ ]:
%%sql
-- 1.7 ANTI JOIN: Clientes de cartera que NUNCA han solicitado un crédito (Distribuidora Andina S.A.S.)
SELECT c.cliente_id, c.nombre, c.ciudad
FROM clientes_cartera c
LEFT JOIN creditos cr ON c.cliente_id = cr.cliente_id
WHERE cr.credito_id IS NULL;

## 2. JOIN con Subconsulta (Derived Table / Tabla Derivada Agregada)
Podemos unir una tabla principal con el resultado de una **Subconsulta en la cláusula `JOIN`**. Esto es extremadamente útil para combinar registros individuales con métricas agregadas (`SUM`, `AVG`, `COUNT`).
**Pregunta de Negocio:** Obtener la información completa de los clientes junto con su resumen consolidado de cartera (total desembolsado y saldo en riesgo).

In [ ]:
%%sql
-- JOIN con Subconsulta Agregada en la cláusula JOIN:
SELECT 
    c.cliente_id,
    c.nombre AS cliente,
    c.ciudad,
    resumen.num_creditos,
    resumen.total_desembolsado,
    resumen.total_saldo_pendiente
FROM clientes_cartera c
JOIN (
    -- Subconsulta que calcula agregaciones por cliente
    SELECT 
        cliente_id,
        COUNT(credito_id) AS num_creditos,
        SUM(monto_desembolsado) AS total_desembolsado,
        SUM(saldo_actual) AS total_saldo_pendiente
    FROM creditos
    GROUP BY cliente_id
) resumen ON c.cliente_id = resumen.cliente_id
ORDER BY resumen.total_saldo_pendiente DESC;

## 3. JOIN con Subconsulta Y Patrón `LIKE` / `ILIKE` 🔥
Ahora combinaremos las tres potentes técnicas: un **`JOIN`** hacia una **Subconsulta en el `FROM`/`JOIN`** que aplica **Filtros por Patrón `LIKE` / `ILIKE`**.
**Pregunta de Negocio:** Encontrar los clientes cuyo correo electrónico pertenezca a dominios corporativos colombianos (`ILIKE '%.co'`) y unir su información con una subconsulta que calcula únicamente el total prestado en créditos con estado de riesgo (`ILIKE '%MORA%'`).

In [ ]:
%%sql
-- JOIN + Subconsulta + LIKE / ILIKE combinados
SELECT 
    c.cliente_id,
    c.nombre AS cliente_corporativo,
    c.email,
    mora_info.creditos_en_mora,
    mora_info.saldo_mora_total
FROM clientes_cartera c
INNER JOIN (
    -- Subconsulta filtrada con LIKE / ILIKE sobre la tabla creditos
    SELECT 
        cliente_id,
        COUNT(credito_id) AS creditos_en_mora,
        SUM(saldo_actual) AS saldo_mora_total
    FROM creditos
    WHERE estado ILIKE '%mora%'
    GROUP BY cliente_id
) mora_info ON c.cliente_id = mora_info.cliente_id
WHERE c.email ILIKE '%.co'
  OR c.nombre ILIKE '%S.A.%';

## 4. Retos Prácticos de JOINs & Subconsultas para el Estudiante 🏋️‍♂️
### ✏️ Reto 1: SELF JOIN con Diferencia Salarial
Consulta los empleados cuyo salario sea **mayor al salario de su jefe directo** usando un `SELF JOIN`.
### ✏️ Reto 2: JOIN con Subconsulta Agregada
Realiza un `LEFT JOIN` entre `clientes_cartera` y una subconsulta que calcule el total de créditos pagados (`estado = 'PAGADO'`). Muestra el nombre del cliente y el monto pagado acumulado.
### ✏️ Reto 3: JOIN + Subconsulta + LIKE
Obtén todos los departamentos de Bogotá (`ciudad ILIKE 'Bogotá'`) unidos con una subconsulta que seleccione el salario promedio de los empleados cuya razón de cargo o nombre contenga la letra `'a'`.

In [ ]:
%%sql
-- 🛠 Espacio para la solución del estudiante:

-- Reto 1: SELECT ... FROM empleados sub JOIN empleados jefe ON ... WHERE sub.salario > jefe.salario

-- Reto 2: SELECT ... FROM clientes_cartera c LEFT JOIN (...) pagados ON ...

-- Reto 3: SELECT ... FROM departamentos d JOIN (...) emp ON ...

In [ ]:
%%sql
-- ==============================================================================
-- 🛠 SOLUCIÓN A LOS RETOS PRÁCTICOS DEL MÓDULO 03 (JOINs & SUBCONSULTAS)
-- ==============================================================================

-- ------------------------------------------------------------------------------
-- ✏️ RETO 1: SELF JOIN con Comparación de Salarios
-- Empleados que ganan MÁS que su propio jefe directo
-- ------------------------------------------------------------------------------
SELECT 
    sub.empleado_id,
    sub.nombre AS subordinado,
    sub.salario AS salario_subordinado,
    jefe.nombre AS jefe_directo,
    jefe.salario AS salario_jefe
FROM empleados sub
JOIN empleados jefe ON sub.jefe_id = jefe.empleado_id
WHERE sub.salario > jefe.salario;


-- ------------------------------------------------------------------------------
-- ✏️ RETO 2: LEFT JOIN con Subconsulta Agregada (Créditos PAGADOS)
-- Clientes y el monto total de créditos que han pagado (0 si no tienen)
-- ------------------------------------------------------------------------------
SELECT 
    c.cliente_id,
    c.nombre AS cliente,
    c.ciudad,
    COALESCE(pagos_info.total_pagado, 0.00) AS total_monto_pagado,
    COALESCE(pagos_info.num_creditos_pagados, 0) AS creditos_pagados
FROM clientes_cartera c
LEFT JOIN (
    -- Subconsulta agregada que calcula totales solo para créditos en estado 'PAGADO'
    SELECT 
        cliente_id,
        COUNT(credito_id) AS num_creditos_pagados,
        SUM(monto_desembolsado) AS total_pagado
    FROM creditos
    WHERE estado = 'PAGADO'
    GROUP BY cliente_id
) pagos_info ON c.cliente_id = pagos_info.cliente_id;


-- ------------------------------------------------------------------------------
-- ✏️ RETO 3: JOIN + Subconsulta + LIKE / ILIKE
-- Departamentos de Bogotá unidos a subconsulta de empleados con la letra 'a' en su nombre
-- ------------------------------------------------------------------------------
SELECT 
    d.depto_id,
    d.nombre AS departamento,
    d.ciudad,
    emp_a.num_empleados,
    ROUND(emp_a.salario_promedio, 2) AS salario_promedio
FROM departamentos d
JOIN (
    -- Subconsulta filtrada por patrón LIKE 'a' y agrupada por departamento
    SELECT 
        depto_id,
        COUNT(empleado_id) AS num_empleados,
        AVG(salario) AS salario_promedio
    FROM empleados
    WHERE nombre ILIKE '%a%'
    GROUP BY depto_id
) emp_a ON d.depto_id = emp_a.depto_id
WHERE d.ciudad ILIKE 'Bogotá';
